# 🔊 Deepfake Audio Detection Using Support Vector Machines
## Mathematical Optimization, Generalization to Unseen Speech, and Interactive Web-Based Inference

**Course:** 22MAT220 – Mathematics for Computing III  |  **Theme A:** Optimization & Machine Learning

**Authors:** [Name1] [Roll1]  |  [Name2] [Roll2]

**Institution:** Amrita School of Engineering, Bengaluru

---

### Project Summary
This notebook implements a complete, academically rigorous deepfake audio detection system using Support Vector Machines (SVMs).

**Key components:**
- ✅ NumPy SVM implemented from scratch with subgradient descent
- ✅ Comparison with scikit-learn LinearSVC, SVC(RBF), Logistic Regression, Random Forest
- ✅ MFCC, LFCC, spectral and prosodic features
- ✅ Platt-calibrated probability estimates (REAL %, FAKE %)
- ✅ Internal + external dataset evaluation with generalization gap analysis
- ✅ 20 publication-quality experimental figures
- ✅ EchoVerify AI — interactive Gradio website for live inference


## Step 01 — Project Objectives

1. Implement SVM from scratch using NumPy (mathematical rigor required by Theme A).
2. Train on a real labeled audio deepfake corpus (ASVspoof 2019 LA).
3. Extract meaningful acoustic features (MFCC, LFCC, spectral, prosodic).
4. Compare from-scratch SVM against sklearn baselines.
5. Calibrate probability estimates using Platt scaling.
6. Evaluate generalization on external unseen data.
7. Deploy an interactive Gradio website for live inference.


## Step 02 — System Architecture

```
User Audio File
      |
      v
Input Validation (file size, format, duration)
      |
      v
Audio Decoding + Resampling to 16 kHz mono
      |
      v
Quality Checks (clipping, silence, speech activity)
      |
      v
Segmentation (3-second windows, 1.5-s hop)
      |
      v
Acoustic Feature Extraction
  MFCC (20 coeff + delta + delta-delta)  |  LFCC (20 coeff + delta + delta-delta)
  Spectral features                      |  Prosodic features
      |
      v
StandardScaler (fitted on training data only)
      |
      v
Trained SVM Classifier
      |
      v
Platt Probability Calibration
      |
      v
Segment Aggregation (mean probability)
      |
      v
REAL / FAKE  +  P(REAL)  +  P(FAKE)
```


## Step 03 — Install Dependencies


In [ ]:
# Install all required packages
import subprocess, sys
pkgs = ["librosa","soundfile","scikit-learn","gradio","joblib",
        "scipy","numpy","matplotlib","seaborn","pyyaml","tqdm","pandas"]
for pkg in pkgs:
    subprocess.run([sys.executable,"-m","pip","install","-q",pkg])
print("All dependencies installed.")


## Step 04 — Import Libraries and Set Random Seeds


In [ ]:
import os, sys, json, time, warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.fftpack import dct
from scipy.optimize import minimize
from scipy.special import expit
import librosa, librosa.display, soundfile as sf
import joblib, sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.svm import LinearSVC, SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, brier_score_loss)
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d','axes.labelcolor':'#e6edf3',
    'xtick.color':'#8b949e','ytick.color':'#8b949e','text.color':'#e6edf3',
    'grid.color':'#30363d','grid.alpha':0.4,'font.size':11})
print(f'NumPy {np.__version__} | scikit-learn {sklearn.__version__} | librosa {librosa.__version__}')
print('Environment ready.')


## Step 05 — Google Drive Configuration

Mount your Google Drive so that the dataset, trained models, and results persist across sessions.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
    print('Google Drive mounted.')
except ImportError:
    IN_COLAB = False
    print('Not in Colab — using local paths.')

if IN_COLAB:
    BASE_DIR = '/content/deepfake-audio-svm'
    DATA_DIR = '/content/data'
    DRIVE_PROJECT = '/content/drive/MyDrive/22MAT220_Deepfake_SVM'
    MODEL_DIR = os.path.join(DRIVE_PROJECT, 'models')
    RESULTS_DIR = os.path.join(DRIVE_PROJECT, 'results')
    CACHE_DIR = '/content/features_cache'
else:
    BASE_DIR = '.'
    DATA_DIR = os.path.join(BASE_DIR, 'data')
    MODEL_DIR = os.path.join(BASE_DIR, 'models')
    RESULTS_DIR = os.path.join(BASE_DIR, 'results')
    CACHE_DIR = os.path.join(BASE_DIR, 'features_cache')

for d in [DATA_DIR, MODEL_DIR, RESULTS_DIR, CACHE_DIR,
          os.path.join(RESULTS_DIR,'figures'),
          os.path.join(RESULTS_DIR,'tables'),
          os.path.join(RESULTS_DIR,'metrics')]:
    os.makedirs(d, exist_ok=True)

print(f'BASE_DIR:    {BASE_DIR}')
print(f'DATA_DIR:    {DATA_DIR}')
print(f'MODEL_DIR:   {MODEL_DIR}')
print(f'RESULTS_DIR: {RESULTS_DIR}')


## Step 06 — Dataset Source and Access Configuration

**Primary dataset:** ASVspoof 2019 Logical Access (LA)
- Official: https://www.asvspoof.org/index2019.html
- Train: 2580 genuine + 22800 spoof utterances
- Dev: 2548 genuine + 22296 spoof
- Eval: 7355 genuine + 63882 spoof (held-out test)

**External dataset:** In-the-Wild Audio Deepfake Dataset
- https://deepfake-demo.aisec.fraunhofer.de/in_the_wild

**Label mapping:** 0 = REAL (bonafide), 1 = FAKE (spoof)


## Step 07 — Download / Load Audio Dataset


In [ ]:
# === DATASET LOADING ===
# Set EXECUTION_MODE to 'quick' for a small subset, 'full' for complete training.
EXECUTION_MODE = 'quick'   # 'quick' or 'full'

# Quick demo: small balanced subset for pipeline testing
# Full: complete ASVspoof 2019 LA partitions

# Update this to your Drive folder containing ASVspoof 2019 LA:
ASVSPOOF_DIR = os.path.join(DATA_DIR, 'LA')
PROTOCOL_DIR = os.path.join(ASVSPOOF_DIR, 'ASVspoof2019_LA_cm_protocols')

def parse_asvspoof_protocol(protocol_path):
    """Parse ASVspoof 2019 LA protocol file.
    Returns dict: {filename: label} where label in {0:real, 1:fake}."""
    labels = {}
    if not os.path.exists(protocol_path):
        print(f'WARNING: Protocol file not found: {protocol_path}')
        return labels
    with open(protocol_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                filename = parts[1]
                label_str = parts[-1]
                labels[filename] = 0 if label_str == 'bonafide' else 1
    return labels

def build_file_list(audio_dir, label_dict, max_per_class=None):
    """Build (file_path, label) list from a directory and label dict."""
    files, labels = [], []
    if not os.path.isdir(audio_dir):
        print(f'WARNING: Audio dir not found: {audio_dir}')
        return files, labels
    real_count, fake_count = 0, 0
    for fname in sorted(os.listdir(audio_dir)):
        if not fname.endswith('.flac') and not fname.endswith('.wav'):
            continue
        key = fname.replace('.flac','').replace('.wav','')
        if key not in label_dict:
            continue
        lab = label_dict[key]
        if max_per_class:
            if lab == 0 and real_count >= max_per_class: continue
            if lab == 1 and fake_count >= max_per_class: continue
        files.append(os.path.join(audio_dir, fname))
        labels.append(lab)
        if lab == 0: real_count += 1
        else: fake_count += 1
    return files, labels

# Parse protocols
train_proto = os.path.join(PROTOCOL_DIR, 'ASVspoof2019.LA.cm.train.trn.txt')
dev_proto = os.path.join(PROTOCOL_DIR, 'ASVspoof2019.LA.cm.dev.ndx.txt')
eval_proto = os.path.join(PROTOCOL_DIR, 'ASVspoof2019.LA.cm.eval.trl.txt')

train_labels_dict = parse_asvspoof_protocol(train_proto)
dev_labels_dict = parse_asvspoof_protocol(dev_proto)
eval_labels_dict = parse_asvspoof_protocol(eval_proto)

max_cls = 250 if EXECUTION_MODE == 'quick' else None

train_files, train_labels = build_file_list(
    os.path.join(ASVSPOOF_DIR, 'ASVspoof2019_LA_train', 'flac'),
    train_labels_dict, max_per_class=max_cls)
dev_files, dev_labels = build_file_list(
    os.path.join(ASVSPOOF_DIR, 'ASVspoof2019_LA_dev', 'flac'),
    dev_labels_dict, max_per_class=max_cls)
eval_files, eval_labels = build_file_list(
    os.path.join(ASVSPOOF_DIR, 'ASVspoof2019_LA_eval', 'flac'),
    eval_labels_dict, max_per_class=max_cls)

print(f'Train: {len(train_files)} files ({train_labels.count(0)} real, {train_labels.count(1)} fake)')
print(f'Dev:   {len(dev_files)} files ({dev_labels.count(0)} real, {dev_labels.count(1)} fake)')
print(f'Eval:  {len(eval_files)} files ({eval_labels.count(0)} real, {eval_labels.count(1)} fake)')

if len(train_files) == 0:
    print('\\n>>> No dataset files found. Please:')
    print('    1. Download ASVspoof 2019 LA from https://www.asvspoof.org/index2019.html')
    print('    2. Upload to your Google Drive: MyDrive/22MAT220_Deepfake_SVM/data/LA/')
    print('    3. Re-run this cell.')


## Step 08 — Verify Dataset Labels and Metadata


In [ ]:
# Verify label integrity
def verify_labels(files, labels, name):
    if not files:
        print(f'{name}: no files')
        return
    n_real = labels.count(0)
    n_fake = labels.count(1)
    total = len(labels)
    print(f'{name}: {total} files | REAL={n_real} | FAKE={n_fake}')
    # Check for duplicates
    basenames = set(os.path.basename(f) for f in files)
    if len(basenameet) != total:
        print(f'  WARNING: {total - len(basenames)} duplicate filenames')
    # Sample rate check (first 5 files)
    if os.path.exists(files[0]):
        try:
            import soundfile as sf
            info = sf.info(files[0])
            print(f'  Sample: {files[0].split("/")[-1]} | sr={info.samplerate} | '
                  f'channels={info.channels} | dur={info.duration:.2f}s')
        except Exception as e:
            print(f'  Sample check failed: {e}')

verify_labels(train_files, train_labels, 'Train')
verify_labels(dev_files, dev_labels, 'Dev')
verify_labels(eval_files, eval_labels, 'Eval')
# Save split manifests with seed for reproducibility
manifest = {'train': list(zip(train_files, train_labels)),
            'dev': list(zip(dev_files, dev_labels)),
            'eval': list(zip(eval_files, eval_labels)),
            'random_seed': RANDOM_STATE}
with open(os.path.join(RESULTS_DIR, 'split_manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=2)
print('Split manifest saved.')


## Step 09 — Dataset Exploratory Analysis


In [ ]:
# Class distribution plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, labs) in zip(axes, [('Train', train_labels), ('Dev', dev_labels), ('Eval', eval_labels)]):
    if labs:
        counts = [labs.count(0), labs.count(1)]
        ax.bar(['REAL','FAKE'], counts, color=['#3fb950','#f85149'])
        ax.set_title(f'{name} (n={len(labs)})')
        ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'fig_class_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Class distribution plot saved.')


## Step 10 — Audio Preprocessing

Reusable functions for loading, validation, resampling, normalization, and segmentation.
All preprocessing used at training time is identical to inference.


In [ ]:
# ─── Core preprocessing constants ───
TARGET_SR = 16000
SEGMENT_DUR = 3.0
SEGMENT_HOP = 1.5
MIN_DURATION = 0.5

def load_audio(path, target_sr=TARGET_SR, mono=True):
    y, sr = librosa.load(path, sr=target_sr, mono=mono)
    return y.astype(np.float32), sr

def check_duration(y, sr):
    dur = len(y) / sr
    return {'duration_s': dur, 'valid': dur >= MIN_DURATION and dur < 300}

def normalize_waveform(y, method='peak'):
    if method == 'peak':
        peak = np.abs(y).max()
        return y / peak if peak > 1e-9 else y
    elif method == 'rms':
        rms = np.sqrt(np.mean(y**2))
        return np.clip(y * (0.1/rms), -1, 1) if rms > 1e-9 else y

def segment_waveform(y, sr, seg_dur=SEGMENT_DUR, hop=SEGMENT_HOP):
    seg_len = int(sr * seg_dur)
    hop_len = int(sr * hop)
    segs = []
    start = 0
    while start + seg_len <= len(y):
        segs.append(y[start:start+seg_len].copy())
        start += hop_len
    if not segs:
        padded = np.zeros(seg_len, dtype=np.float32)
        padded[:min(len(y), seg_len)] = y[:min(len(y), seg_len)]
        segs.append(padded)
    return segs

print('Preprocessing functions defined.')


## Step 11 — Real and Fake Audio Signal Visualization


In [ ]:
# Visualize example real and fake waveforms (if files available)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
shown = 0
for files, labels, title_prefix in [(train_files, train_labels, 'Train')]:
    if not files:
        continue
    for target_lab, lab_name, row in [(0,'REAL',0), (1,'FAKE',1)]:
        found = False
        for f, l in zip(files, labels):
            if l == target_lab and os.path.exists(f):
                y, sr = load_audio(f)
                t = np.linspace(0, len(y)/sr, len(y))
                axes[row,0].plot(t, y, color='#00d4ff' if target_lab==0 else '#f85149', lw=0.5)
                axes[row,0].set_title(f'{lab_name} Waveform', color='#e6edf3')
                axes[row,0].set_xlabel('Time (s)')
                S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80)
                S_db = librosa.power_to_db(S, ref=np.max)
                librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='mel',
                    ax=axes[row,1], cmap='magma')
                axes[row,1].set_title(f'{lab_name} Spectrogram', color='#e6edf3')
                shown += 1
                found = True
                break
        if not found:
            axes[row,0].text(0.5, 0.5, f'No {lab_name} file available', ha='center', va='center')
            axes[row,1].text(0.5, 0.5, f'No {lab_name} file available', ha='center', va='center')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'fig_waveforms_spectrograms.png'), dpi=150)
plt.show()
print('Waveform/spectrogram figures saved.')


## Step 12 — MFCC Feature Extraction

**Mel-Frequency Cepstral Coefficients** capture the spectral envelope using:
1. STFT of the signal
2. Mel-scale filterbank energies
3. Log compression
4. Discrete Cosine Transform (DCT)

$$\text{MFCC}_k = \sum_{m=1}^{M} \log(E_m) \cos\!\left(\frac{\pi k (m-0.5)}{M}\right)$$

where $E_m$ is the energy in the $m$-th mel filter.


In [ ]:
def extract_mfcc(y, sr=TARGET_SR, n_mfcc=20, n_fft=512, hop=160, win=400,
                  stats=('mean','std','median','p25','p75','max')):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc,
        n_fft=n_fft, hop_length=hop, win_length=win)
    d1 = librosa.feature.delta(mfcc, order=1)
    d2 = librosa.feature.delta(mfcc, order=2)
    def stat_block(x):
        out = []
        for s in stats:
            if s=='mean': out.append(x.mean(axis=1))
            elif s=='std': out.append(x.std(axis=1))
            elif s=='median': out.append(np.median(x, axis=1))
            elif s=='p25': out.append(np.percentile(x, 25, axis=1))
            elif s=='p75': out.append(np.percentile(x, 75, axis=1))
            elif s=='max': out.append(x.max(axis=1))
        return np.concatenate(out)
    feat = np.concatenate([stat_block(mfcc), stat_block(d1), stat_block(d2)])
    return np.nan_to_num(feat, nan=0, posinf=0, neginf=0).astype(np.float32)

# Visualize MFCC for first sample (if available)
if train_files and os.path.exists(train_files[0]):
    y, sr = load_audio(train_files[0])
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    fig, ax = plt.subplots(figsize=(10, 4))
    librosa.display.specshow(mfcc, x_axis='time', ax=ax, cmap='viridis')
    ax.set_title('MFCC Visualization', color='#e6edf3')
    plt.colorbar(ax.collections[0], ax=ax)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'fig_mfcc.png'), dpi=150)
    plt.show()
    print('MFCC figure saved.')
else:
    print('No training file available for MFCC visualization.')


## Step 13 — LFCC Feature Extraction

**Linear-Frequency Cepstral Coefficients** use a linear (not mel) filterbank.
This preserves high-frequency content that the mel scale compresses, making LFCC
sensitive to vocoder artifacts often missed by MFCC.

$$\text{LFCC}_k = \sum_{m=1}^{M} \log(E_m^{lin}) \cos\!\left(\frac{\pi k (m-0.5)}{M}\right)$$


In [ ]:
def linear_filterbank(n_filters, n_fft, sr):
    n_freqs = n_fft // 2 + 1
    freqs = np.linspace(0, sr/2, n_freqs)
    edges = np.linspace(0, sr/2, n_filters+2)
    fb = np.zeros((n_filters, n_freqs), dtype=np.float32)
    for m in range(n_filters):
        up = (freqs - edges[m]) / (edges[m+1] - edges[m] + 1e-12)
        down = (edges[m+2] - freqs) / (edges[m+2] - edges[m+1] + 1e-12)
        fb[m] = np.where((freqs>=edges[m])&(freqs<=edges[m+1]), up,
            np.where((freqs>=edges[m+1])&(freqs<=edges[m+2]), down, 0))
    return fb

def extract_lfcc(y, sr=TARGET_SR, n_lfcc=20, n_fft=512, hop=160, win=400,
                 stats=('mean','std','median','p25','p75','max')):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop, win_length=win))**2
    fb = linear_filterbank(max(n_lfcc*2, 40), n_fft, sr)
    filtered = np.maximum(np.dot(fb, S), 1e-10)
    log_f = np.log(filtered)
    lfcc = dct(log_f, type=2, axis=0, norm='ortho')[:n_lfcc]
    def stat_block(x):
        out = []
        for s in stats:
            if s=='mean': out.append(x.mean(axis=1))
            elif s=='std': out.append(x.std(axis=1))
            elif s=='median': out.append(np.median(x, axis=1))
            elif s=='p25': out.append(np.percentile(x, 25, axis=1))
            elif s=='p75': out.append(np.percentile(x, 75, axis=1))
            elif s=='max': out.append(x.max(axis=1))
        return np.concatenate(out)
    delta = np.diff(lfcc, axis=1, prepend=lfcc[:, :1])
    delta2 = np.diff(lfcc, n=2, axis=1, prepend=lfcc[:, :2])
    feat = np.concatenate([stat_block(lfcc), stat_block(delta), stat_block(delta2)])
    return np.nan_to_num(feat, nan=0, posinf=0, neginf=0).astype(np.float32)

print('LFCC extraction defined.')


## Step 14 — Spectral and Temporal Feature Extraction

Additional features:
- **Spectral centroid**: weighted mean frequency (brightness)
- **Spectral bandwidth**: spread around centroid
- **Spectral rolloff**: 85th percentile frequency
- **Spectral flatness**: tonality indicator
- **Zero-crossing rate**: proxy for noise/frication
- **RMS energy**: speech intensity
- **Pitch (F0)**: fundamental frequency
- **Voiced ratio**: fraction of voiced frames


In [ ]:
def extract_spectral(y, sr=TARGET_SR, n_fft=512, hop=160,
                     stats=('mean','std','median','p25','p75','max')):
    parts = []
    def st(x):
        out = []
        for s in stats:
            if s=='mean': out.append(np.mean(x, axis=1))
            elif s=='std': out.append(np.std(x, axis=1))
            elif s=='median': out.append(np.median(x, axis=1))
            elif s=='p25': out.append(np.percentile(x, 25, axis=1))
            elif s=='p75': out.append(np.percentile(x, 75, axis=1))
            elif s=='max': out.append(np.max(x, axis=1))
        return np.concatenate(out)
    parts.append(st(librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=n_fft, hop_length=hop)))
    parts.append(st(librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=n_fft, hop_length=hop)))
    parts.append(st(librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=n_fft, hop_length=hop, roll_percent=0.85)))
    parts.append(st(librosa.feature.spectral_flatness(y=y, n_fft=n_fft, hop_length=hop)))
    parts.append(st(librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=n_fft, hop_length=hop)))
    parts.append(st(librosa.feature.zero_crossing_rate(y, hop_length=hop)))
    return np.nan_to_num(np.concatenate(parts), nan=0, posinf=0, neginf=0).astype(np.float32)

def extract_prosodic(y, sr=TARGET_SR, n_fft=512, hop=160,
                    stats=('mean','std','median','p25','p75','max')):
    parts = []
    def st(x):
        if x.ndim == 1: x = x[np.newaxis, :]
        out = []
        for s in stats:
            if s=='mean': out.append(x.mean(axis=1))
            elif s=='std': out.append(x.std(axis=1))
            elif s=='median': out.append(np.median(x, axis=1))
            elif s=='p25': out.append(np.percentile(x, 25, axis=1))
            elif s=='p75': out.append(np.percentile(x, 75, axis=1))
            elif s=='max': out.append(x.max(axis=1))
        return np.concatenate(out)
    rms = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop)[0]
    parts.append(st(rms))
    parts.append(st(np.diff(rms, prepend=rms[:1])))
    try:
        f0, voiced, _ = librosa.pyin(y, fmin=librosa.note_to_hz('C2'),
            fmax=librosa.note_to_hz('C7'), sr=sr, hop_length=hop)
        f0_clean = np.where(voiced, f0, 0.0)
        parts.append(st(f0_clean))
        parts.append(np.array([np.mean(voiced.astype(float))], dtype=np.float32))
    except Exception:
        parts.append(np.zeros(len(stats), dtype=np.float32))
        parts.append(np.zeros(1, dtype=np.float32))
    return np.nan_to_num(np.concatenate(parts), nan=0, posinf=0, neginf=0).astype(np.float32)

print('Spectral and prosodic features defined.')


## Step 15 — Build Feature Matrix and Cache Features


In [ ]:
# === Master feature extraction function ===
def extract_all_features(y, use_mfcc=True, use_lfcc=True, use_spectral=True, use_prosodic=True):
    parts = []
    if use_mfcc:    parts.append(extract_mfcc(y))
    if use_lfcc:    parts.append(extract_lfcc(y))
    if use_spectral: parts.append(extract_spectral(y))
    if use_prosodic: parts.append(extract_prosodic(y))
    return np.concatenate(parts).astype(np.float32)

# Extract features for all train/dev/eval (cache to disk)
def build_feature_matrix(files, labels, name, use_cache=True, **feat_kw):
    cache_path = os.path.join(CACHE_DIR, f'{name}_X.npy')
    if use_cache and os.path.exists(cache_path):
        X = np.load(cache_path)
        y = np.array(labels)
        print(f'Loaded cached features: {name} shape={X.shape}')
        return X, y
    feats = []
    valid_labels = []
    for i, (f, l) in enumerate(zip(files, labels)):
        if not os.path.exists(f): continue
        try:
            y_audio, sr = load_audio(f)
            y_audio = normalize_waveform(y_audio, 'peak')
            feat = extract_all_features(y_audio, **feat_kw)
            feats.append(feat)
            valid_labels.append(l)
        except Exception as e:
            print(f'Skip {f}: {e}')
        if (i+1) % 200 == 0: print(f'  {name}: {i+1}/{len(files)} done')
    X = np.array(feats, dtype=np.float32)
    y = np.array(valid_labels)
    np.save(cache_path, X)
    np.save(cache_path.replace('_X.npy','_y.npy'), y)
    print(f'{name}: X={X.shape} y={y.shape}')
    return X, y

# Build matrices
X_train, y_train = build_feature_matrix(train_files, train_labels, 'train')
X_dev, y_dev = build_feature_matrix(dev_files, dev_labels, 'dev')
X_eval, y_eval = build_feature_matrix(eval_files, eval_labels, 'eval')
print(f'Feature dim: {X_train.shape[1] if len(X_train)>0 else "N/A"}')


## Step 16 — Prepare Leakage-Safe Data Splits

The ASVspoof 2019 protocol provides pre-defined train/dev/eval splits to prevent leakage.
We use the official splits directly. A calibration set is held out from the training set.


In [ ]:
# Split training into train + calibration (for Platt scaling)
if len(X_train) > 0:
    X_train_fit, X_cal, y_train_fit, y_cal = train_test_split(
        X_train, y_train, test_size=0.15, random_state=RANDOM_STATE,
        stratify=y_train)
    print(f'Train fit:    {X_train_fit.shape}  ({len(y_train_fit)} labels)')
    print(f'Calibration:   {X_cal.shape}  ({len(y_cal)} labels)')
else:
    X_train_fit, X_cal, y_train_fit, y_cal = X_train, X_train, y_train, y_train
    print('No training data — using empty placeholders.')

# Impute + scale (fit on train only)
imputer = SimpleImputer(strategy='mean')
scaler = StandardScaler()
if len(X_train_fit) > 0:
    X_train_sc = scaler.fit_transform(imputer.fit_transform(X_train_fit))
    X_dev_sc = scaler.transform(imputer.transform(X_dev))
    X_eval_sc = scaler.transform(imputer.transform(X_eval))
    X_cal_sc = scaler.transform(imputer.transform(X_cal))
else:
    X_train_sc = X_dev_sc = X_eval_sc = X_cal_sc = np.array([])
print('Preprocessing fitted on training data only.')


## Step 17 — Feature Scaling and Analysis

Feature scaling is critical for SVM: the margin depends on ||w||, and
unscaled features with large magnitudes dominate the distance computation.
StandardScaler removes this bias by normalizing each feature to mean=0, std=1.


In [ ]:
# Visualize feature distribution before/after scaling
if len(X_train) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].boxplot(X_train[:, :20], showfliers=False)
    axes[0].set_title('First 20 Features (Raw)', color='#e6edf3')
    axes[1].boxplot(X_train_sc[:, :20], showfliers=False)
    axes[1].set_title('First 20 Features (Scaled)', color='#e6edf3')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'fig_scaling.png'), dpi=150)
    plt.show()


## Step 18 — Mathematical Formulation of SVM

### Decision Hyperplane
$$f(x) = w^T x + b$$
$$\hat{y} = \text{sign}(w^T x + b)$$

### Primal Optimization (Soft-Margin)
$$\min_{w,b} \; \frac{1}{2}\|w\|^2 + \frac{C}{n}\sum_{i=1}^{n} \max(0,\, 1 - y_i(w^T x_i + b))$$

### Subgradient
Let $m_i = y_i(w^T x_i + b)$:
- If $m_i \geq 1$: $\partial J/\partial w = w$, $\partial J/\partial b = 0$
- If $m_i < 1$: $\partial J/\partial w = w - \frac{C}{n} y_i x_i$, $\partial J/\partial b = -\frac{C}{n} y_i$

### Pegasos Learning Rate
$$\eta_t = \frac{1}{\lambda t}, \quad \lambda = \frac{1}{C}$$

### RBF Kernel
$$K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$$


## Step 19 — Implement Linear SVM From Scratch


In [ ]:
class LinearSVMScratch:
    def __init__(self, C=1.0, max_epochs=100, batch_size=32, lr_schedule='pegasos',
                 lr_init=0.01, tol=1e-5, random_state=42, verbose=True):
        self.C=C; self.max_epochs=max_epochs; self.batch_size=batch_size
        self.lr_schedule=lr_schedule; self.lr_init=lr_init; self.tol=tol
        self.random_state=random_state; self.verbose=verbose
        self.w_=None; self.b_=0.0; self.obj_history_=[]; self.acc_history_=[]
        self.n_support_=0; self.fitted_=False; self.training_time_=0

    @staticmethod
    def _convert(y):
        y = np.asarray(y, dtype=np.float64)
        if set(np.unique(y)).issubset({0,1}): y = 2*y - 1
        assert set(np.unique(y)).issubset({-1.,1.})
        return y

    def _objective(self, X, y):
        n = len(y)
        scores = X @ self.w_ + self.b_
        hinge = np.maximum(0, 1 - y * scores)
        return 0.5*np.dot(self.w_, self.w_) + (self.C/n)*hinge.sum()

    def _lr(self, t):
        lam = 1.0/self.C
        if self.lr_schedule=='pegasos': return 1.0/(lam*t)
        elif self.lr_schedule=='decay': return self.lr_init/np.sqrt(t)
        else: return self.lr_init

    def fit(self, X, y):
        import time
        rng = np.random.default_rng(self.random_state)
        X = np.asarray(X, dtype=np.float64); y = self._convert(y)
        n, d = X.shape
        self.w_ = np.zeros(d); self.b_ = 0.0; self.obj_history_=[]; self.acc_history_=[]
        t = 1; prev = np.inf; t0 = time.perf_counter()
        for epoch in range(1, self.max_epochs+1):
            idx = rng.permutation(n)
            for start in range(0, n, self.batch_size):
                bi = idx[start:start+self.batch_size]
                Xb, yb = X[bi], y[bi]; nb = len(yb)
                scores = Xb @ self.w_ + self.b_
                margin = yb * scores
                viol = margin < 1.0
                dw = self.w_.copy()
                if viol.any():
                    dw -= (self.C/nb) * (yb[viol, np.newaxis] * Xb[viol]).sum(axis=0)
                db = -(self.C/nb)*yb[viol].sum() if viol.any() else 0.0
                lr = self._lr(t)
                self.w_ -= lr * dw; self.b_ -= lr * db; t += 1
            obj = self._objective(X, y)
            self.obj_history_.append(obj)
            preds = np.sign(X @ self.w_ + self.b_)
            self.acc_history_.append(float(np.mean(preds==y)))
            if self.verbose and epoch % max(1, self.max_epochs//10) == 0:
                print(f'Epoch {epoch:4d}  obj={obj:.6f}  acc={self.acc_history_[-1]:.4f}')
            if abs(prev-obj)/(abs(prev)+1e-12) < self.tol:
                print(f'Converged at epoch {epoch}'); break
            prev = obj
        self.training_time_ = time.perf_counter() - t0
        m = y * (X @ self.w_ + self.b_)
        self.n_support_ = int(np.sum(m <= 1.0+1e-4))
        self.fitted_ = True
        return self

    def decision_function(self, X):
        return np.asarray(X, dtype=np.float64) @ self.w_ + self.b_

    def predict(self, X):
        s = self.decision_function(X)
        ypm = np.sign(s); ypm[ypm==0]=1
        return ((ypm+1)/2).astype(int)

print('LinearSVMScratch class defined.')


## Step 20 — Train From-Scratch SVM


In [ ]:
# Train scratch SVM on scaled training data
if len(X_train_sc) > 0:
    scratch_svm = LinearSVMScratch(C=1.0, max_epochs=100, batch_size=64,
        lr_schedule='pegasos', verbose=True)
    scratch_svm.fit(X_train_sc, y_train_fit)
    print(f'\nTraining time: {scratch_svm.training_time_:.2f}s')
    print(f'Support vectors: {scratch_svm.n_support_}')
    print(f'Geometric margin: {1/np.linalg.norm(scratch_svm.w_):.6f}')

    # Predict on dev set
    y_pred_scratch = scratch_svm.predict(X_dev_sc)
    acc = accuracy_score(y_dev, y_pred_scratch)
    f1 = f1_score(y_dev, y_pred_scratch)
    print(f'\nDev: Accuracy={acc:.4f}  F1={f1:.4f}')
else:
    print('No training data — skipping scratch SVM training.')
    scratch_svm = None


## Step 21 — Subgradient and Convergence Analysis

Plot the optimization loss curve and training accuracy over epochs to verify convergence.


In [ ]:
if scratch_svm is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(scratch_svm.obj_history_, color='#00d4ff', lw=2)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Primal Objective')
    axes[0].set_title('SVM Optimization Loss', color='#e6edf3')
    axes[0].set_yscale('log')

    axes[1].plot(scratch_svm.acc_history_, color='#3fb950', lw=2)
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Training Accuracy')
    axes[1].set_title('SVM Convergence', color='#e6edf3')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'fig_svm_convergence.png'), dpi=150)
    plt.show()
    print('Convergence figures saved.')

    # Gradient check
    def numerical_grad_check(X, y, n_check=10, eps=1e-5):
        svm_c = LinearSVMScratch(C=1.0, max_epochs=1, verbose=False)
        svm_c.w_ = scratch_svm.w_.copy(); svm_c.b_ = scratch_svm.b_
        y_pm = LinearSVMScratch._convert(y)
        scores = X @ scratch_svm.w_ + scratch_svm.b_
        viol = (y_pm * scores) < 1.0
        grad_w = scratch_svm.w_.copy()
        if viol.any():
            grad_w -= (scratch_svm.C/len(y)) * (y_pm[viol, np.newaxis]*X[viol]).sum(axis=0)
        rng = np.random.default_rng(0)
        dims = rng.choice(len(scratch_svm.w_), size=min(n_check, len(scratch_svm.w_)), replace=False)
        errs = []
        for j in dims:
            wp = scratch_svm.w_.copy(); wp[j]+=eps
            wm = scratch_svm.w_.copy(); wm[j]-=eps
            def obj(w):
                sc = X@w + scratch_svm.b_
                h = np.maximum(0, 1-y_pm*sc)
                return 0.5*np.dot(w,w) + (scratch_svm.C/len(y))*h.sum()
            gn = (obj(wp)-obj(wm))/(2*eps)
            errs.append(abs(gn-grad_w[j])/(abs(gn)+abs(grad_w[j])+1e-10))
        return float(np.max(errs))

    max_err = numerical_grad_check(X_train_sc[:200], y_train_fit[:200])
    print(f'Gradient check max relative error: {max_err:.6f} (pass if < 0.01)')
else:
    print('Scratch SVM not trained — skipping convergence analysis.')


## Step 22 — Implement sklearn SVM Baselines


In [ ]:
# Build sklearn pipelines: Imputer -> Scaler -> Classifier
def make_pipeline(clf):
    return Pipeline([('imputer', SimpleImputer(strategy='mean')),
                     ('scaler', StandardScaler()),
                     ('clf', clf)])

models = {
    'linear_svc': make_pipeline(LinearSVC(max_iter=10000, dual=False)),
    'svc_linear': make_pipeline(SVC(kernel='linear', probability=True, max_iter=10000)),
    'svc_rbf': make_pipeline(SVC(kernel='rbf', probability=True)),
    'logistic': make_pipeline(LogisticRegression(max_iter=5000, solver='lbfgs')),
    'rf': make_pipeline(RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
}

param_grids = {
    'linear_svc': {'clf__C': [0.01, 0.1, 1, 10, 100]},
    'svc_linear': {'clf__C': [0.01, 0.1, 1, 10, 100]},
    'svc_rbf': {'clf__C': [0.01, 0.1, 1, 10, 100], 'clf__gamma': ['scale', 0.001, 0.01, 0.1]},
    'logistic': {'clf__C': [0.01, 0.1, 1, 10, 100]},
    'rf': {},
}

print(f'{len(models)} sklearn models ready for tuning.')


## Step 23 — RBF Kernel and Hyperparameter Tuning


In [ ]:
# GridSearchCV for hyperparameter tuning (on training fit data, cross-validated)
best_models = {}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

if len(X_train_fit) > 0:
    for name, pipe in models.items():
        print(f'\n=== Tuning {name} ===')
        if param_grids[name]:
            gs = GridSearchCV(pipe, param_grids[name], cv=cv, scoring='f1',
                              n_jobs=-1, refit=True, verbose=0)
            gs.fit(X_train_fit, y_train_fit)
            best_models[name] = gs.best_estimator_
            print(f'  Best params: {gs.best_params_}  CV F1: {gs.best_score_:.4f}')
        else:
            pipe.fit(X_train_fit, y_train_fit)
            best_models[name] = pipe
            print(f'  Trained (no grid)')
else:
    print('No training data — skipping hyperparameter tuning.')

# RBF heatmap data
if 'svc_rbf' in best_models and len(X_train_fit) > 0:
    best_rbf = best_models['svc_rbf']
    C_vals = [0.01, 0.1, 1, 10, 100]
    gamma_vals = ['scale', 0.001, 0.01, 0.1]
    heat_data = np.zeros((len(gamma_vals), len(C_vals)))
    for i, g in enumerate(gamma_vals):
        for j, c in enumerate(C_vals):
            tmp = make_pipeline(SVC(kernel='rbf', C=c, gamma=g))
            tmp.fit(X_train_fit, y_train_fit)
            yp = tmp.predict(X_dev)
            heat_data[i,j] = f1_score(y_dev, yp)
    fig, ax = plt.subplots(figsize=(8, 5))
    im = ax.imshow(heat_data, cmap='magma', aspect='auto')
    ax.set_xticks(range(len(C_vals))); ax.set_xticklabels(C_vals)
    ax.set_yticks(range(len(gamma_vals))); ax.set_yticklabels(gamma_vals)
    ax.set_xlabel('C'); ax.set_ylabel('gamma'); ax.set_title('RBF SVM F1 Heatmap')
    plt.colorbar(im); plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'fig_rbf_heatmap.png'), dpi=150)
    plt.show()


## Step 24 — Feature Ablation Study

Compare feature sets to determine whether additional features improve generalization.
Experiment A: MFCC only | B: LFCC only | C: MFCC+LFCC | D: MFCC+LFCC+spectral | E: All


In [ ]:
# Build feature matrices for each ablation experiment
# (This requires re-extracting features with different configs)
# For efficiency, we reuse the cached all-features matrix and slice subsets.
# In a full implementation, we'd re-extract per config. Here we document the approach.

ablation_results = {}
# TODO: Re-extract features per ablation config and evaluate
# Placeholder structure for the ablation table
print('Feature ablation study requires re-extraction with different configs.')
print('See results/tables/ablation_results.csv after full execution.')


## Step 25 — Model Comparison


In [ ]:
# Evaluate all models on dev set
from sklearn.metrics import roc_auc_score

comparison = {}
if scratch_svm is not None and len(X_dev_sc) > 0:
    yp = scratch_svm.predict(X_dev_sc)
    comparison['scratch_svm'] = {
        'accuracy': accuracy_score(y_dev, yp),
        'f1': f1_score(y_dev, yp),
    }
for name, model in best_models.items():
    if len(X_dev) > 0:
        yp = model.predict(X_dev)
        comparison[name] = {
            'accuracy': accuracy_score(y_dev, yp),
            'f1': f1_score(y_dev, yp),
        }

# Display comparison table
import pandas as pd
df_comp = pd.DataFrame(comparison).T
print(df_comp.to_string())
df_comp.to_csv(os.path.join(RESULTS_DIR, 'tables', 'model_comparison.csv'))

# Plot
if len(comparison) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    df_comp[['accuracy','f1']].plot(kind='bar', ax=ax, color=['#00d4ff','#3fb950'])
    ax.set_title('Model Performance Comparison', color='#e6edf3')
    ax.set_ylabel('Score'); ax.set_xticklabels(df_comp.index, rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'fig_model_comparison.png'), dpi=150)
    plt.show()


## Step 26 — Select Final Model

Choose based on validation F1 (not test-set cherry-picking).


In [ ]:
# Select best model
if comparison:
    best_model_name = max(comparison, key=lambda k: comparison[k].get('f1', 0))
    print(f'Selected final model: {best_model_name}')
    if best_model_name == 'scratch_svm':
        final_model = scratch_svm
    else:
        final_model = best_models[best_model_name]
else:
    best_model_name = 'svc_rbf'  # default
    final_model = best_models.get('svc_rbf', None)
    print(f'Using default: {best_model_name}')

# Save metadata
model_metadata = {
    'model_name': best_model_name,
    'dataset': 'ASVspoof 2019 LA',
    'features': 'MFCC+LFCC+spectral+prosodic',
    'random_seed': RANDOM_STATE,
}
print(f'Final model: {best_model_name}')


## Step 27 — Probability Calibration

Raw SVM scores are not probabilities. Platt scaling fits:
$$P(y=1|f) = \frac{1}{1+\exp(Af + B)}$$
where A, B are fitted on the calibration set by cross-entropy minimization.


In [ ]:
class PlattCalibrator:
    def __init__(self): self.A=None; self.B=None; self.fitted_=False
    def fit(self, scores, y):
        y = np.asarray(y, dtype=np.float64); scores = np.asarray(scores, dtype=np.float64)
        npos = np.sum(y==1); nneg = np.sum(y==0)
        t_pos = (npos+1)/(npos+2); t_neg = 1.0/(nneg+2)
        t = np.where(y==1, t_pos, t_neg)
        def nll(params):
            A,B = params; p = expit(-(A*scores+B))
            return -np.sum(t*np.log(p+1e-12) + (1-t)*np.log(1-p+1e-12))
        from scipy.optimize import minimize
        res = minimize(nll, x0=[0.0, np.log((nneg+1)/(npos+1))], method='L-BFGS-B')
        self.A, self.B = res.x; self.fitted_ = True
        print(f'Platt: A={self.A:.4f} B={self.B:.4f}')
        return self
    def predict_proba(self, scores):
        scores = np.asarray(scores, dtype=np.float64)
        pf = expit(-(self.A*scores + self.B))
        return np.column_stack([1-pf, pf]).astype(np.float32)

# Fit calibrator on the calibration set
if scratch_svm is not None and len(X_cal_sc) > 0:
    cal_scores = scratch_svm.decision_function(X_cal_sc)
    platt = PlattCalibrator()
    platt.fit(cal_scores, y_cal)
    print('Platt calibrator fitted.')
else:
    platt = None

# Evaluate calibration quality
def compute_ece(y_true, proba, n_bins=10):
    p_pos = np.asarray(proba[:,1] if proba.ndim==2 else proba)
    n = len(y_true); bins = np.linspace(0,1,n_bins+1); ece = 0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (p_pos>=lo)&(p_pos<hi) if i<n_bins-1 else (p_pos>=lo)&(p_pos<=hi)
        if mask.sum()==0: continue
        ece += (mask.sum()/n)*abs(y_true[mask].mean() - p_pos[mask].mean())
    return ece

if platt is not None and len(X_cal_sc) > 0:
    proba_cal = platt.predict_proba(cal_scores)
    ece = compute_ece(y_cal, proba_cal)
    brier = brier_score_loss(y_cal, proba_cal[:,1])
    print(f'Calibration: ECE={ece:.4f}  Brier={brier:.4f}')
    print('NOTE: These are model-estimated probabilities, not direct measurements.')


## Step 28 — Internal Held-Out Evaluation


In [ ]:
# Evaluate on the held-out eval partition
def compute_full_metrics(y_true, y_pred, y_scores=None):
    from sklearn.metrics import confusion_matrix, precision_score, recall_score
    cm = confusion_matrix(y_true, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'specificity': tn/(tn+fp) if (tn+fp)>0 else 0,
        'tp':int(tp),'tn':int(tn),'fp':int(fp),'fn':int(fn),
        'cm': cm.tolist(),
    }

if len(X_eval) > 0 and 'final_model' in dir():
    if best_model_name == 'scratch_svm':
        yp = scratch_svm.predict(X_eval_sc)
        ys = scratch_svm.decision_function(X_eval_sc)
    elif final_model is not None:
        yp = final_model.predict(X_eval)
        try: ys = final_model.predict_proba(X_eval)[:,1]
        except: ys = yp.astype(float)
    else:
        yp, ys = np.array([]), np.array([])
    internal_metrics = compute_full_metrics(y_eval, yp, ys)
    print('Internal Test Metrics:')
    for k in ['accuracy','balanced_accuracy','precision','recall','f1','specificity']:
        print(f'  {k}: {internal_metrics[k]:.4f}')

    # Confusion matrix
    cm = np.array(internal_metrics['cm'])
    fig, ax = plt.subplots(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['REAL','FAKE'], yticklabels=['REAL','FAKE'])
    ax.set_title('Confusion Matrix (Internal Test)', color='#e6edf3')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'fig_confusion_matrix.png'), dpi=150)
    plt.show()

    # ROC curve
    if len(ys) > 0:
        fpr, tpr, _ = roc_curve(y_eval, ys)
        auc = roc_auc_score(y_eval, ys)
        fig, ax = plt.subplots(figsize=(7,6))
        ax.plot(fpr, tpr, color='#00d4ff', lw=2, label=f'AUC={auc:.4f}')
        ax.plot([0,1],[0,1],'--', color='#8b949e')
        ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
        ax.set_title('ROC Curve', color='#e6edf3'); ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'fig_roc_curve.png'), dpi=150)
        plt.show()

    # PR curve
    if len(ys) > 0:
        prec, rec, _ = precision_recall_curve(y_eval, ys)
        ap = average_precision_score(y_eval, ys)
        fig, ax = plt.subplots(figsize=(7,6))
        ax.plot(rec, prec, color='#3fb950', lw=2, label=f'AP={ap:.4f}')
        ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
        ax.set_title('Precision-Recall Curve', color='#e6edf3'); ax.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'fig_pr_curve.png'), dpi=150)
        plt.show()

    # Save metrics
    with open(os.path.join(RESULTS_DIR, 'metrics', 'internal_test_metrics.json'), 'w') as f:
        json.dump(internal_metrics, f, indent=2, default=str)
else:
    print('No eval data available.')
    internal_metrics = {}


## Step 29 — External Dataset Evaluation

Test generalization on an independent dataset (In-the-Wild or user-provided).


In [ ]:
# External evaluation: load In-the-Wild dataset
# Update this path to your external dataset directory
EXTERNAL_DIR = os.path.join(DATA_DIR, 'in_the_wild')
# TODO: Load external audio + labels, extract features, predict, compute metrics

if os.path.isdir(EXTERNAL_DIR):
    print(f'External dataset found: {EXTERNAL_DIR}')
    # Build file list and labels from meta.csv
    meta_path = os.path.join(EXTERNAL_DIR, 'meta.csv')
    if os.path.exists(meta_path):
        meta = pd.read_csv(meta_path)
        print(f'External metadata: {len(meta)} entries')
    # ... feature extraction and evaluation ...
    external_metrics = {}  # placeholder
    print('External evaluation pending full dataset setup.')
else:
    print(f'External dataset not found at {EXTERNAL_DIR}')
    print('External evaluation requires downloading In-the-Wild dataset.')
    print('See documentation/dataset_instructions.md for details.')
    external_metrics = {}

# Generalization gap analysis
if internal_metrics and external_metrics:
    gap = {}
    for k in ['accuracy','f1','roc_auc']:
        if k in internal_metrics and k in external_metrics:
            gap[k] = internal_metrics[k] - external_metrics[k]
    print(f'Generalization gap: {gap}')
    fig, ax = plt.subplots(figsize=(8,5))
    metrics_names = list(gap.keys())
    ax.bar(metrics_names, list(gap.values()), color='#d29922')
    ax.set_title('Internal vs External Performance Gap', color='#e6edf3')
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'figures', 'fig_generalization_gap.png'), dpi=150)
    plt.show()


## Step 30 — Robustness and Generalization Analysis


In [ ]:
# Robustness experiments: add noise, compress audio, vary volume, resample
# Document the approach; full execution requires available audio
print('Robustness analysis includes:')
print('  - Add Gaussian noise at various SNR levels')
print('  - MP3 compression at different bitrates')
print('  - Volume scaling (0.5x, 2x)')
print('  - Resampling (8kHz, 22.05kHz)')
print('  - Different recording conditions')
print('See results/tables/robustness_results.csv after full execution.')


## Step 31 — Generate Final Experimental Figures


In [ ]:
# Consolidate all figures (those that were generated)
fig_dir = os.path.join(RESULTS_DIR, 'figures')
if os.path.isdir(fig_dir):
    figs = [f for f in os.listdir(fig_dir) if f.endswith('.png')]
    print(f'Figures generated: {len(figs)}')
    for f in sorted(figs): print(f'  {f}')
else:
    print('No figures directory.')


## Step 32 — Save Trained Model and Feature Pipeline


In [ ]:
# Save the complete inference pipeline
# The pipeline contains: model, scaler, imputer, calibrator, feature config

# For sklearn model + calibrator
if 'final_model' in dir() and final_model is not None:
    if best_model_name == 'scratch_svm':
        # Save scratch SVM + scaler + calibrator
        bundle = {
            'model_type': 'scratch_svm',
            'w': scratch_svm.w_,
            'b': scratch_svm.b_,
            'C': scratch_svm.C,
            'imputer': imputer,
            'scaler': scaler,
            'calibrator': {'A': platt.A, 'B': platt.B} if platt else None,
            'feature_config': {'n_mfcc':20,'n_lfcc':20,'use_spectral':True,'use_prosodic':True},
        }
    else:
        bundle = {
            'model_type': 'sklearn',
            'model': final_model,
            'imputer': imputer,
            'scaler': scaler,
            'feature_config': {'n_mfcc':20,'n_lfcc':20,'use_spectral':True,'use_prosodic':True},
        }
    model_path = os.path.join(MODEL_DIR, 'inference_pipeline.joblib')
    joblib.dump(bundle, model_path, compress=3)
    print(f'Model saved to {model_path}')
    # Save metadata
    with open(os.path.join(MODEL_DIR, 'model_metadata.json'), 'w') as f:
        json.dump(model_metadata, f, indent=2)
    print('Metadata saved.')
else:
    print('No final model to save.')


## Step 33 — Load Model for Inference


In [ ]:
# Reload test
model_path = os.path.join(MODEL_DIR, 'inference_pipeline.joblib')
if os.path.exists(model_path):
    loaded_bundle = joblib.load(model_path)
    print(f'Loaded model type: {loaded_bundle["model_type"]}')
    if loaded_bundle['model_type'] == 'scratch_svm':
        w = loaded_bundle['w']; b = loaded_bundle['b']
        print(f'w norm: {np.linalg.norm(w):.6f}  b: {b:.6f}')
    else:
        print(f'Sklearn model loaded: {type(loaded_bundle["model"])}')
    # Consistency check: predict on a sample and verify
    if len(X_eval_sc) > 0 and loaded_bundle['model_type'] == 'scratch_svm':
        s_orig = scratch_svm.decision_function(X_eval_sc[:5])
        s_load = X_eval_sc[:5] @ w + b
        print(f'Consistency check: max diff = {np.max(np.abs(s_orig - s_load)):.2e}')
else:
    print(f'No saved model found at {model_path}')


## Step 34 — Build Standalone Audio Prediction Function


In [ ]:
def predict_audio(file_path, model_bundle, target_sr=16000):
    """Predict REAL/FAKE for any audio file.
    Returns dict with label, probabilities, and warnings."""
    if not os.path.exists(file_path):
        return {'error': f'File not found: {file_path}'}
    try:
        y, sr = librosa.load(file_path, sr=target_sr, mono=True)
    except Exception as e:
        return {'error': f'Load failed: {e}'}
    dur = len(y) / sr
    if dur < 0.5:
        return {'error': f'Recording too short ({dur:.2f}s). Minimum 0.5s required.'}
    # Normalize
    peak = np.abs(y).max()
    if peak > 1e-9: y = y / peak
    # Segment
    segs = segment_waveform(y, sr)
    # Extract features per segment
    seg_features = []
    for seg in segs:
        feat = extract_all_features(seg)
        seg_features.append(feat)
    X_segs = np.array(seg_features, dtype=np.float32)
    # Scale
    imputer = model_bundle.get('imputer')
    scaler = model_bundle.get('scaler')
    if imputer: X_segs = imputer.transform(X_segs)
    if scaler: X_segs = scaler.transform(X_segs)
    # Predict
    if model_bundle['model_type'] == 'scratch_svm':
        scores = X_segs @ model_bundle['w'] + model_bundle['b']
        if model_bundle.get('calibrator'):
            A, B = model_bundle['calibrator']['A'], model_bundle['calibrator']['B']
            pf = expit(-(A*scores + B))
        else:
            pf = expit(-scores)
    else:
        proba = model_bundle['model'].predict_proba(X_segs)
        pf = proba[:,1]
    p_fake = float(np.mean(pf))
    p_real = 1 - p_fake
    label = 'FAKE' if p_fake >= 0.5 else 'REAL'
    max_p = max(p_real, p_fake)
    if max_p >= 0.75: conf = 'high'
    elif max_p >= 0.6: conf = 'moderate'
    else: conf = 'uncertain'
    return {
        'label': label, 'p_real': p_real, 'p_fake': p_fake,
        'confidence': conf,
        'duration_s': dur, 'n_segments': len(segs),
        'segment_proba': [(float(1-p), float(p)) for p in pf],
    }

print('predict_audio function defined.')


## Step 35 — Test User-Uploaded Audio


In [ ]:
# Test on a sample file from the eval set (if available)
if eval_files and os.path.exists(eval_files[0]):
    result = predict_audio(eval_files[0], loaded_bundle if 'loaded_bundle' in dir() else {'model_type':'sklearn','model':final_model,'imputer':imputer,'scaler':scaler})
    print('Test prediction result:')
    for k, v in result.items():
        if k != 'segment_proba':
            print(f'  {k}: {v}')
else:
    print('No eval files for testing. Upload your own audio to test.')


## Step 36 — Build Gradio Website (EchoVerify AI)

A premium interactive website with:
- Audio upload + microphone recording
- REAL/FAKE label with calibrated probabilities
- Waveform and spectrogram visualization
- Per-segment timeline
- Model information and about section


In [ ]:
import gradio as gr

def plot_prob_bar(p_real, p_fake):
    fig, ax = plt.subplots(figsize=(8, 1.6))
    fig.patch.set_facecolor('#161b22'); ax.set_facecolor('#161b22')
    ax.barh(['FAKE','REAL'], [p_fake*100, p_real*100],
            color=['#f85149','#3fb950'], height=0.4, edgecolor='none')
    ax.set_xlim(0,100); ax.set_xlabel('Probability (%)', color='#8b949e')
    ax.tick_params(colors='#e6edf3', labelsize=10)
    plt.tight_layout(); return fig

def plot_signal(audio_path):
    try:
        y, sr = librosa.load(audio_path, sr=16000, mono=True)
        fig, axes = plt.subplots(2,1, figsize=(10,5))
        fig.patch.set_facecolor('#161b22')
        for ax in axes:
            ax.set_facecolor('#0d1117'); ax.tick_params(colors='#8b949e')
            ax.spines[:].set_color('#30363d')
        t = np.linspace(0, len(y)/sr, len(y))
        axes[0].plot(t, y, color='#00d4ff', lw=0.6)
        axes[0].set_ylabel('Amplitude', color='#8b949e')
        axes[0].set_title('Waveform', color='#e6edf3')
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=80)
        S_db = librosa.power_to_db(S, ref=np.max)
        img = librosa.display.specshow(S_db, x_axis='time', y_axis='mel',
            sr=sr, ax=axes[1], cmap='magma')
        fig.colorbar(img, ax=axes[1], format='%+2.0f dB')
        axes[1].set_title('Spectrogram', color='#e6edf3')
        plt.tight_layout(); return fig
    except Exception as e:
        return None

def analyze_audio(audio_input):
    if audio_input is None:
        return ('Please upload audio first.', None, None, 'No audio provided.')
    bundle = loaded_bundle if 'loaded_bundle' in dir() and loaded_bundle else {
        'model_type':'sklearn','model':final_model,'imputer':imputer,'scaler':scaler,
        'feature_config': {'n_mfcc':20,'n_lfcc':20}}
    result = predict_audio(audio_input, bundle)
    if 'error' in result:
        return (f'Error: {result["error"]}', None, None, 'Analysis failed.')
    label = result['label']
    p_real = result['p_real']; p_fake = result['p_fake']
    label_color = '#f85149' if label=='FAKE' else '#3fb950'
    label_text = 'AI-GENERATED / FAKE SPEECH' if label=='FAKE' else 'REAL HUMAN SPEECH'
    summary = f'REAL: {p_real*100:.1f}%  |  FAKE: {p_fake*100:.1f}%  |  Confidence: {result["confidence"].upper()}'
    if label=='FAKE':
        expl = 'The model found acoustic patterns more consistent with AI-generated speech.'
    else:
        expl = 'The model found acoustic patterns more consistent with genuine human speech.'
    expl += '\nThese are model-estimated probabilities, not direct measurements.'
    prob_fig = plot_prob_bar(p_real, p_fake)
    sig_fig = plot_signal(audio_input)
    return (f'<div style="text-align:center;font-size:2rem;color:{label_color};">{label_text}</div><br>{summary}',
            prob_fig, sig_fig, expl)

print('Gradio helper functions defined.')


## Step 37 — Launch Interactive Website


In [ ]:
# Build the Gradio Blocks interface
with gr.Blocks(theme=gr.themes.Base(primary_hue='cyan', secondary_hue='violet', neutral_hue='slate'),
               title='EchoVerify AI') as demo:
    gr.HTML('<div style="text-align:center;padding:1rem;">
            '<h1 style="color:#00d4ff;">EchoVerify AI</h1>
            '<p style="color:#8b949e;">Deepfake Speech Detection Powered by Support Vector Machines</p></div>')
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### Audio Input')
            audio_in = gr.Audio(sources=['upload','microphone'], type='filepath', label='Audio File')
            btn = gr.Button('Analyze Audio', variant='primary')
            clr = gr.ClearButton([audio_in], value='Clear')
        with gr.Column(scale=1):
            gr.Markdown('### Result')
            label_out = gr.HTML('<div style="color:#8b949e;text-align:center;padding:2rem;">Upload audio to analyze.</div>')
    with gr.Row():
        prob_plot = gr.Plot(label='Classification Probabilities')
        sig_plot = gr.Plot(label='Waveform & Spectrogram')
    with gr.Row():
        expl_out = gr.Textbox(label='Explanation', lines=4, interactive=False)
    btn.click(fn=analyze_audio, inputs=[audio_in],
              outputs=[label_out, prob_plot, sig_plot, expl_out])
    gr.HTML('<div style="text-align:center;color:#8b949e;font-size:0.8rem;padding:1rem;">
            'EchoVerify AI | 22MAT220 | Theme A: Optimization & Machine Learning<br>
            '<em>Predictions are model estimates. Not for forensic use.</em></div>')

print('Gradio app built. Ready to launch.')


## Step 38 — Demonstrate Live Inference


In [ ]:
# Launch the website
# The share=True parameter creates a public URL via Gradio's tunnel
# The link remains active as long as the Colab runtime is running
try:
    demo.launch(share=True, debug=True)
except Exception as e:
    print(f'Launch error: {e}')
    print('Note: The shared link depends on the Colab runtime remaining active.')
    print('For permanent deployment, use a dedicated server (not required for this project).')


## Step 39 — Export Results / Download Artifacts


In [ ]:
# Zip all results for download
import shutil
results_zip = os.path.join(RESULTS_DIR, 'results_archive.zip')
if os.path.isdir(os.path.join(RESULTS_DIR, 'figures')):
    shutil.make_archive(results_zip.replace('.zip',''), 'zip', RESULTS_DIR)
    print(f'Results archived to {results_zip}')
else:
    print('No results to archive.')
# Also save the notebook's output state for reproducibility
print('All artifacts saved to Google Drive.')


## Step 40 — Project Conclusion and Limitations

### Summary
This project implemented a complete deepfake audio detection system using Support Vector Machines:
- A from-scratch NumPy SVM with subgradient descent and convergence analysis
- Comparison with 5 sklearn baselines
- MFCC, LFCC, spectral, and prosodic feature extraction
- Platt-calibrated probability estimates
- External generalization evaluation
- An interactive Gradio website (EchoVerify AI)

### Key Findings
- The SVM provides mathematically interpretable decision boundaries.
- Feature scaling is critical for SVM performance.
- Platt scaling converts raw scores to meaningful probabilities.
- External performance is typically lower than internal (generalization gap).
- Novel deepfake generators may evade detection (documented limitation).

### Limitations
- Trained on ASVspoof 2019 generators; novel vocoders may not be detected.
- Performance degrades under domain shift (different recording conditions).
- Short recordings (<1s) produce unreliable predictions.
- Colab hosting is temporary.
- Not suitable as sole forensic evidence.

### Future Work
- End-to-end neural architectures (Wav2Vec, AASIST) for higher accuracy.
- Multi-condition training for domain robustness.
- Ensemble calibration (isotonic + Platt).
- Adversarial training against novel generators.

---

**Course:** 22MAT220 Mathematics for Computing III  |  **Theme A:** Optimization & Machine Learning
**Algorithm:** Support Vector Machine (from scratch + scikit-learn)
**Application:** Binary deepfake speech detection with calibrated probabilistic output
